In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Data Preprocessing

In [2]:
data = pd.read_csv("/content/mail_data.csv")

In [3]:
df = data.where((pd.notnull(data)),'')

In [4]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
df['Category'] = df['Category'].str.lower().replace('ham', 'not spam')

In [6]:
df.head()

,Category,Message
0,not spam,"Go until jurong point, crazy.. Available only ..."
1,not spam,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,not spam,U dun say so early hor... U c already then say...
4,not spam,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.shape

(5572, 2)

In [8]:
df.loc[df['Category'] == 'spam', 'Category',] = 0
df.loc[df['Category'] == 'not spam', 'Category',] = 1

spam -> 0

not span -> 1

In [9]:
X = df['Message']
y = df['Category']

# Train Test Split

In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=3)

In [11]:
X.shape, X_train.shape, X_test.shape

((5572,), (4457,), (1115,))

# Feature Extraction

In [12]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english", lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

Y_train = Y_train.astype(int)
Y_test = Y_test.astype(int)


In [13]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 42569 stored elements and shape (4457, 7360)>

# Training The Model

## Logistic Regression

In [14]:
model = LogisticRegression(max_iter=1000)

In [15]:
model.fit(X_train_tfidf, Y_train)

LogisticRegression(max_iter=1000)

In [16]:
Y_pred = model.predict(X_test_tfidf)

In [17]:
print("Accuracy Score: ", accuracy_score(Y_test, Y_pred))

Accuracy Score:  0.9704035874439462


In [18]:
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       1.00      0.79      0.88       155
           1       0.97      1.00      0.98       960

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [19]:
print(confusion_matrix(Y_test, Y_pred))

[[122  33]
 [  0 960]]


# Testing the Model

In [20]:
email_text = ["Congratulations! You have won a free gift voucher. Click the link now to claim your reward."]

email_tfidf = tfidf_vectorizer.transform(email_text)

email_prediction = model.predict(email_tfidf)
print(email_prediction)

if email_prediction[0] == 1:
    print("Not Spam mail")
else:
    print("Spam mail")


[0]
Spam mail


# Saving the Model

In [ ]:
import joblib

joblib.dump(model, "logistic_regression_model.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']